# Crime proxy scores from `US_Crime_DataSet.csv` + `uscities.csv`

**Inputs**
- `US_Crime_DataSet.csv` — FBI-style **incident** records (homicides / manslaughter, 1980–2014).
- `uscities.csv` — US cities with **population** and a space-separated **`zips`** field.

**Approach**
1. Restrict to a recent window (`YEAR_START`–`YEAR_END`) and count homicides by **City + State**.
2. Join to `uscities` on normalized `city_ascii` + `state_name` (many agency/city names won’t match — that’s expected).
3. Compute **annualized murder rate per 100k**: `homicides / years / population * 100_000`.
4. Expand each city’s `zips` into one row per **5-digit ZIP**.
5. Map rate → **`crime_proxy`** in \[0, 1\] with min–max scaling (optionally winsorize outliers).

**Output** — `crime_proxy_by_zip.csv` — columns suitable for loading in the API (join on ZIP).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# --- paths (repo root = parent of notebooks/) ---
ROOT = Path("..").resolve()
CRIME_CSV = ROOT / "US_Crime_DataSet.csv"
CITIES_CSV = ROOT / "uscities.csv"
OUT_CSV = ROOT / "backend" / "data" / "crime_proxy_by_zip.csv"

YEAR_START, YEAR_END = 2005, 2014  # 10-year window; adjust as needed
YEARS = YEAR_END - YEAR_START + 1

# Drop tiny places (unstable rates / city-name collisions with crime agencies)
MIN_POPULATION = 5_000
# Plausible upper bound for annual murder rate per 100k (US city context); tune if needed
MAX_MURDER_RATE_PER_100K_ANNUAL = 75.0

## 1) Inspect schemas

In [ ]:
crime = pd.read_csv(CRIME_CSV, low_memory=False, nrows=2000)
print("US_Crime_DataSet columns:", list(crime.columns))
print(crime.dtypes.head(12))

cities = pd.read_csv(CITIES_CSV, nrows=5)
print("\nuscities columns:", list(cities.columns))

## 2) Aggregate homicides by city / state

In [ ]:
crime_full = pd.read_csv(CRIME_CSV, low_memory=False)
crime_full = crime_full[
    (crime_full["Year"] >= YEAR_START) & (crime_full["Year"] <= YEAR_END)
].copy()

crime_full["city_key"] = (
    crime_full["City"].astype(str).str.strip().str.lower()
    + "|"
    + crime_full["State"].astype(str).str.strip().str.lower()
)

homicides = (
    crime_full.groupby("city_key", as_index=False)
    .agg(
        city=("City", "first"),
        state=("State", "first"),
        homicides=("Record ID", "count"),
    )
)
print(f"City groups in window: {len(homicides):,}")
homicides.head()

## 3) Join `uscities` (population + zips)

In [ ]:
uc = pd.read_csv(CITIES_CSV, low_memory=False)
uc["city_key"] = (
    uc["city_ascii"].astype(str).str.strip().str.lower()
    + "|"
    + uc["state_name"].astype(str).str.strip().str.lower()
)

merged = homicides.merge(
    uc[
        [
            "city_key",
            "city_ascii",
            "state_id",
            "state_name",
            "county_fips",
            "population",
            "lat",
            "lng",
            "zips",
        ]
    ],
    on="city_key",
    how="inner",
)
merged["population"] = pd.to_numeric(merged["population"], errors="coerce")
merged = merged[merged["population"] > 0]
merged = merged[merged["population"] >= MIN_POPULATION]

merged["murder_rate_per_100k_annual"] = (
    merged["homicides"] / YEARS / merged["population"] * 100_000.0
)
# Cap extreme values before scaling (residual noise from homonym cities / agency vs municipality)
merged["murder_rate_per_100k_annual"] = merged["murder_rate_per_100k_annual"].clip(
    upper=MAX_MURDER_RATE_PER_100K_ANNUAL
)

print(f"Matched cities (pop>={MIN_POPULATION:,}): {len(merged):,} (of {len(homicides):,} crime city groups)")
merged[["city_ascii", "state_id", "population", "homicides", "murder_rate_per_100k_annual"]].head(10)

## 4) Winsorize + scale to `crime_proxy` \[0, 1\]

Rates are already **population-filtered** and **hard-capped**. Then winsorize at the 99th percentile and min–max to \[0, 1\].

In [ ]:
rate = merged["murder_rate_per_100k_annual"].clip(
    upper=merged["murder_rate_per_100k_annual"].quantile(0.99)
)
rmin, rmax = rate.min(), rate.max()
merged["crime_proxy"] = (rate - rmin) / (rmax - rmin + 1e-9)
merged["crime_proxy"] = merged["crime_proxy"].clip(0, 1)
print("rate min/max (after clip):", float(rmin), float(rmax))

## 5) Expand `zips` → one row per ZIP5

In [ ]:
def split_zips(z: str) -> list[str]:
    if pd.isna(z):
        return []
    parts = str(z).split()
    out = []
    for p in parts:
        digits = "".join(c for c in p if c.isdigit())
        if len(digits) >= 5:
            out.append(digits[:5])
    return out


rows = []
for _, r in merged.iterrows():
    for z in split_zips(r["zips"]):
        rows.append(
            {
                "zip_code": z,
                "state_id": r["state_id"],
                "city_ascii": r["city_ascii"],
                "county_fips": r["county_fips"],
                "population_city": int(r["population"]),
                "homicides_window": int(r["homicides"]),
                "years_window": YEARS,
                "murder_rate_per_100k_annual": round(float(r["murder_rate_per_100k_annual"]), 4),
                "crime_proxy": round(float(r["crime_proxy"]), 6),
            }
        )

zip_df = pd.DataFrame(rows)
# If a ZIP appears in multiple city rows, keep the max crime signal (conservative for risk)
zip_df = zip_df.sort_values("crime_proxy", ascending=False).drop_duplicates("zip_code", keep="first")
print(f"Unique ZIPs: {len(zip_df):,}")
zip_df.head(12)

## 6) Save CSV

In [ ]:
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
zip_df.to_csv(OUT_CSV, index=False)
print("Wrote:", OUT_CSV)
print(zip_df["crime_proxy"].describe())

## Notes

- **Coverage**: Only ZIPs whose **city name + state** matched `uscities` get a row. Others need a fallback (e.g. state average or hash) in the app.
- **Signal**: This file reflects **homicide/manslaughter** rates from your incident extract, not total UCR Part I crime — label the column accordingly in product copy.
- **Join in Python**: `pd.read_csv("crime_proxy_by_zip.csv")` then `df[df.zip_code == zip5]` or merge on ZIP.